## 1. Data Collection and Initial Inspection

The Food Delivery Times dataset is used as a reference logistics dataset for this data preprocessing task.

The dataset contains 1,000 food delivery records and 9 columns. It includes information about delivery distance, weather, traffic level, time of day, vehicle type, preparation time, courier experience, and delivery time.

The dataset is loaded into Python using the Pandas library. An initial inspection is performed to understand the size and structure of the dataset before starting the data cleaning process.

In [ ]:
import pandas as pd

df = pd.read_csv('/content/Food_Delivery_Times.csv')

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1000, 9)


,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68


## 2. Data Types and Missing Value Check

After loading the dataset, the next step is to check the data types of each column and identify missing values.

Checking data types helps us understand whether each variable is stored correctly, such as numeric or categorical data. Missing values are also identified because incomplete data can affect the accuracy of further analysis and modeling.

In [ ]:
# Check data types
print("Data Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Data Types:
Order_ID                    int64
Distance_km               float64
Weather                    object
Traffic_Level              object
Time_of_Day                object
Vehicle_Type               object
Preparation_Time_min        int64
Courier_Experience_yrs    float64
Delivery_Time_min           int64
dtype: object

Missing Values:
Order_ID                   0
Distance_km                0
Weather                   30
Traffic_Level             30
Time_of_Day               30
Vehicle_Type               0
Preparation_Time_min       0
Courier_Experience_yrs    30
Delivery_Time_min          0
dtype: int64


## 3. Checking Missing Values

Before cleaning the data, I checked how many values are missing in each column.

This is important because missing data can affect the analysis. I also calculated the percentage of missing values to understand how much data is missing.

In [ ]:
missing_percentage = df.isnull().sum() / len(df) * 100

print(missing_percentage.round(2))

Order_ID                  0.0
Distance_km               0.0
Weather                   3.0
Traffic_Level             3.0
Time_of_Day               3.0
Vehicle_Type              0.0
Preparation_Time_min      0.0
Courier_Experience_yrs    3.0
Delivery_Time_min         0.0
dtype: float64


## 4. Checking Duplicate Records

I checked the dataset for duplicate rows.

Duplicate records can give extra importance to the same data and may affect the analysis, so they should be identified before continuing with the preprocessing.

In [ ]:
duplicates = df.duplicated().sum()

print("Number of duplicate rows:", duplicates)

Number of duplicate rows: 0


## 5. Handling Missing Values

The dataset has some missing values in Weather, Traffic_Level, Time_of_Day, and Courier_Experience_yrs.

I used the mode for categorical columns and the median for Courier_Experience_yrs. These methods are simple and help keep the dataset complete without removing rows.

In [ ]:
# Fill missing categorical values with mode
for column in ["Weather", "Traffic_Level", "Time_of_Day"]:
    df[column] = df[column].fillna(df[column].mode()[0])

# Fill missing numerical values with median
df["Courier_Experience_yrs"] = df["Courier_Experience_yrs"].fillna(
    df["Courier_Experience_yrs"].median()
)

print(df.isnull().sum())

Order_ID                  0
Distance_km               0
Weather                   0
Traffic_Level             0
Time_of_Day               0
Vehicle_Type              0
Preparation_Time_min      0
Courier_Experience_yrs    0
Delivery_Time_min         0
dtype: int64


## 6. Checking for Outliers

I checked the numerical columns for unusual values using the IQR method.

Outliers are values that are very different from most of the other values. They can affect analysis, so it is useful to identify them before continuing.

In [ ]:
numeric_columns = [
    "Distance_km",
    "Preparation_Time_min",
    "Courier_Experience_yrs",
    "Delivery_Time_min"
]

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower) | (df[column] > upper)]

    print(column, ":", len(outliers), "outliers")

Distance_km : 0 outliers
Preparation_Time_min : 0 outliers
Courier_Experience_yrs : 0 outliers
Delivery_Time_min : 6 outliers


## 7. Handling Outliers

The IQR method identified some outliers in the numerical columns. I decided not to remove them because they may represent real delivery situations, such as longer delivery times.

Keeping these values avoids losing potentially useful information from the dataset.

In [ ]:
# Keep the identified outliers
print("Outliers were identified but not removed.")
print("Dataset shape:", df.shape)

Outliers were identified but not removed.
Dataset shape: (1000, 9)


## 7. Outlier Decision

The IQR method identified 6 outliers in Delivery_Time_min.

I did not remove these values because they may represent genuine cases where a delivery took longer than usual. Removing them without checking could result in loss of useful information.

Therefore, the outliers were identified and reviewed, but they were kept in the dataset.

## 8. Normalizing Numerical Data

The numerical columns have different ranges, so I used Min-Max normalization to bring them to a common scale between 0 and 1.

This can make the data easier to use in further analysis and machine learning.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

numeric_columns = [
    "Distance_km",
    "Preparation_Time_min",
    "Courier_Experience_yrs",
    "Delivery_Time_min"
]

normalized_df = df.copy()

scaler = MinMaxScaler()

normalized_df[numeric_columns] = scaler.fit_transform(
    normalized_df[numeric_columns]
)

normalized_df[numeric_columns].head()

,Distance_km,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,0.378351,0.291667,0.111111,0.241379
1,0.815979,0.625000,0.222222,0.524138
2,0.460309,0.958333,0.111111,0.351724
3,0.353093,0.000000,0.111111,0.200000
4,0.950515,0.458333,0.555556,0.413793


In [ ]:
normalized_df.to_csv(
    '/content/normalized_food_delivery_data.csv',
    index=False
)

print("Normalized dataset saved successfully.")

Normalized dataset saved successfully.


## 9. Final Data Check

After cleaning and normalization, I checked the dataset again to make sure there are no missing values and the numerical columns are properly normalized.

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDataset shape:", df.shape)

print("\nNormalized numerical data:")
print(df[numeric_columns].head())

Missing values:
Order_ID                  0
Distance_km               0
Weather                   0
Traffic_Level             0
Time_of_Day               0
Vehicle_Type              0
Preparation_Time_min      0
Courier_Experience_yrs    0
Delivery_Time_min         0
dtype: int64

Dataset shape: (1000, 9)

Normalized numerical data:
   Distance_km  Preparation_Time_min  Courier_Experience_yrs  \
0         7.93                    12                     1.0   
1        16.42                    20                     2.0   
2         9.52                    28                     1.0   
3         7.44                     5                     1.0   
4        19.03                    16                     5.0   

   Delivery_Time_min  
0                 43  
1                 84  
2                 59  
3                 37  
4                 68  


## 10. Saving the Cleaned Data

After completing the preprocessing steps, I saved the cleaned dataset as a new CSV file.

This keeps the original dataset unchanged and gives us a prepared file that can be used for further analysis.

In [ ]:
df.to_csv('/content/cleaned_food_delivery_data.csv', index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


## 11. Preprocessing Summary

The dataset was prepared using a few basic preprocessing steps. Missing values were filled using mode and median, duplicate rows were checked, and outliers were identified using the IQR method.

The identified outliers were kept because they may represent genuine delivery records. Finally, numerical columns were normalized using Min-Max normalization.

The cleaned dataset is now ready for further analysis.

In [ ]:
print("Preprocessing completed successfully.")
print("Final dataset shape:", df.shape)
print("Total missing values:", df.isnull().sum().sum())

Preprocessing completed successfully.
Final dataset shape: (1000, 9)
Total missing values: 0


## 12. Final Dataset Preview

I checked the cleaned dataset one final time to make sure the preprocessing was completed correctly.

The dataset has no missing values, no duplicate rows, and the numerical columns have been normalized.

In [ ]:
df.head()

,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68


## 13. Reflection

Data quality is important in logistics because incorrect or incomplete data can affect the results of analysis.

In this project, missing values were handled, duplicate records were checked, and outliers were identified. Normalization was also applied to numerical data.

These preprocessing steps make the dataset more suitable for further analysis and can help in making better decisions related to delivery operations and logistics planning.